# Week 7 Assignment - Functions

Run the code cell below (click the play button or press **Shift+Enter**).
Answer each prompt in the box that appears. Type `no` when asked to add another employee to finish.
A file called `employees.json` will be created — find it in the folder icon on the left sidebar.

In [1]:
# Week 7 Assignment - Functions
# This program is a refactored version of the Week 6 program.
# It gathers employee information with validation, stores it in a list of
# dictionaries, updates the data using comprehensions, and finally writes
# the data out to a JSON file.
#
# The Week 6 logic was full of repeated input/validation loops. In this
# version that redundant code has been replaced with reusable functions:
#   - one generic function gathers and re-prompts for input
#   - small validator functions check each individual field
#   - each comprehension lives in its own function
#   - a dedicated function saves the final list to a JSON file


# json is part of the Python standard library and is used to write the
# employee list out to a .json file at the end of the program.
import json


# ---------- CONSTANTS ----------
# These rule sets used to be defined inline. Defining them once at the top
# removes duplication and makes the validation rules easy to find and change.

# Characters that are NOT allowed in the email field
INVALID_EMAIL_CHARS = ['!', '"', "'", '#', '$', '%', '^', '&', '*', '(', ')',
                       '=', '+', ',', '<', '>', '/', '?', ';', ':', '[', ']',
                       '{', '}', '\\']

# Characters that are NOT allowed in the address field
INVALID_ADDRESS_CHARS = ['!', '"', "'", '@', '$', '%', '^', '&', '*', '_',
                         '=', '+', '<', '>', '?', ';', ':', '[', ']', '{', '}']

# Special characters that ARE allowed in the name field
ALLOWED_NAME_CHARS = [' ', "'", '-']


# ---------- VALIDATOR FUNCTIONS ----------
# Each validator takes the raw text the user typed and returns a tuple:
#   (is_valid, cleaned_value, error_message)
# This lets a validator both check a value AND convert it (e.g. text -> int).
# Keeping each rule in its own function removes the repeated checking logic
# that appeared over and over in the Week 6 version.

def validate_id(value):
    """ID must be all digits and 7 characters or less."""
    if value.isdigit() and len(value) <= 7:
        return True, int(value), ""  # convert to a number on success
    return False, None, "Invalid ID. Must be a number with 7 digits or less."


def validate_name(value):
    """Name may only contain letters, spaces, apostrophes, and hyphens."""
    for char in value:
        if not char.isalpha() and char not in ALLOWED_NAME_CHARS:
            return False, None, "Invalid Name. Only letters, spaces, ' and - are allowed."
    return True, value, ""


def validate_email(value):
    """Email must not contain any of the invalid characters."""
    for char in value:
        if char in INVALID_EMAIL_CHARS:
            return False, None, "Invalid Email. It contains characters that are not allowed."
    return True, value, ""


def validate_address(value):
    """Address must not contain any of the invalid characters."""
    for char in value:
        if char in INVALID_ADDRESS_CHARS:
            return False, None, "Invalid Address. It contains characters that are not allowed."
    return True, value, ""


def validate_salary(value):
    """Salary must be a number between 18 and 27 (inclusive)."""
    try:
        salary = float(value)
    except ValueError:
        return False, None, "Invalid Salary. Please enter a number."
    if 18 <= salary <= 27:
        return True, salary, ""  # return it as a float on success
    return False, None, "Salary must be between 18 and 27."


# ---------- GENERIC INPUT FUNCTION ----------
# This single function replaces the many nearly-identical "while True" input
# loops from Week 6. It keeps asking until the user provides a valid value,
# using whichever validator function is passed in. The 'required' flag
# controls whether an empty entry is allowed (used for the optional address).

def get_input(prompt, validator, required=True):
    """Prompt the user repeatedly until a valid value is entered, then return it."""
    while True:
        raw = input(prompt)

        # Handle an empty entry
        if raw == "":
            if required:
                print("This field is required. Please try again.")
                continue
            return ""  # optional field was skipped, return empty string

        # Run the field-specific validator
        is_valid, cleaned_value, error_message = validator(raw)
        if is_valid:
            return cleaned_value

        # Validation failed, show the reason and ask again
        print(error_message)


# ---------- YES / NO HELPER ----------
# Replaces the repeated yes/no prompting loop. Returns True for yes, False for no.

def ask_yes_no(prompt):
    """Ask a yes/no question and return True for yes, False for no."""
    while True:
        answer = input(prompt).lower()
        if answer in ("yes", "y"):
            return True
        if answer in ("no", "n"):
            return False
        print("Please enter yes or no.")


# ---------- GATHER ONE EMPLOYEE ----------
# Builds a single employee dictionary by calling get_input for each field.

def get_employee():
    """Collect all fields for one employee and return them as a dictionary."""
    emp_id = get_input("Enter Employee ID (number, 7 digits or less): ", validate_id)
    emp_name = get_input("Enter Employee Name: ", validate_name)
    emp_email = get_input("Enter Employee Email Address: ", validate_email)
    emp_address = get_input(
        "Enter Employee Address (optional, press Enter to skip): ",
        validate_address,
        required=False,
    )
    emp_salary = get_input("Enter Employee Salary (between 18 and 27): ", validate_salary)

    return {
        "id": emp_id,
        "name": emp_name,
        "email": emp_email,
        "address": emp_address,
        "salary": emp_salary,
    }


# ---------- GATHER ALL EMPLOYEES ----------
# Runs the main collection loop, adding employees until the user is done.

def gather_employees():
    """Collect employees one at a time until the user chooses to stop."""
    employees = []
    while True:
        employees.append(get_employee())
        print("Employee added successfully!\n")

        if not ask_yes_no("Do you want to add another employee? (yes/no): "):
            break  # user said no, stop collecting

    return employees


# ---------- COMPREHENSION FUNCTIONS ----------
# Each comprehension from Week 6 now lives inside its own function.

def add_department(employees):
    """Comprehension 1: append ' IT Department' to every employee's name."""
    return [{**emp, "name": emp["name"] + " IT Department"} for emp in employees]


def apply_benefits(employees):
    """Comprehension 2: increase every employee's salary by 30% for benefits."""
    return [{**emp, "salary": emp["salary"] * 1.30} for emp in employees]


# ---------- SAVE TO JSON ----------
# Writes the final list of employee dictionaries out to a JSON file.

def save_to_json(employees, filename="employees.json"):
    """Write the employee list to a JSON file."""
    with open(filename, "w") as json_file:
        json.dump(employees, json_file, indent=4)
    print(f"\nEmployee data successfully saved to {filename}")


# ---------- MAIN PROGRAM ----------
# Orchestrates the whole program by calling the functions in order.

def main():
    # Step 1: gather all employee data from the user
    employees = gather_employees()

    # Step 2: run the comprehensions to update the data
    employees = add_department(employees)
    employees = apply_benefits(employees)

    # Step 3: print the updated list
    print("\n--- Updated Employee List ---")
    for emp in employees:
        print(emp)

    # Step 4: write the data out to a JSON file
    save_to_json(employees)


# Standard Python entry point so main() only runs when this file is executed
# directly (not when it is imported by another file).
if __name__ == "__main__":
    main()


Enter Employee ID (number, 7 digits or less): 1234567
Enter Employee Name: Test
Enter Employee Email Address: test@abc.com
Enter Employee Address (optional, press Enter to skip): 
Enter Employee Salary (between 18 and 27): 22
Employee added successfully!

Do you want to add another employee? (yes/no): no

--- Updated Employee List ---
{'id': 1234567, 'name': 'Test IT Department', 'email': 'test@abc.com', 'address': '', 'salary': 28.6}

Employee data successfully saved to employees.json


## Optional: download the JSON file to your computer
Run the cell below after the program finishes if you want to save `employees.json` locally.

In [2]:
from google.colab import files
files.download('employees.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>